# 04 — From evaluation manifest to promotion

**10–15 minute lab.** Load the shared core manifest, run a deterministic baseline and candidate,
compare compatible quality axes, and promote a complete clean run.

**Flow:** manifest → baseline/candidate → comparison → promotion.


In [ ]:
from pathlib import Path

from raglab.evaluation import EvaluationApplication, HermeticEvaluationExecutor, load_manifest

artifact_dir = Path("/tmp/raglab-short-evaluation-lab")

def metadata():
    return {
        "commit": "hermetic-notebook",
        "dirty": False,
        "generation_model": "hermetic-generator",
        "embedding_model": "hermetic-embedding",
        "hardware": {"kind": "deterministic"},
        "hardware_fingerprint": "deterministic-hardware",
    }


## Checkpoint 1 — Objective: inspect the evaluation contract

**Run:** load the same versioned manifest used by `raglab-evaluate`.


In [ ]:
manifest = load_manifest(profile="core")
print(f"Profile: {manifest.profile} (schema {manifest.schema_version})")
print("Sources:", [source.id for source in manifest.sources])
print("Cases:", [case.id for case in manifest.cases])
print(f"Chunk checks: {len(manifest.must_separate)} separate, {len(manifest.must_keep)} keep")


### What to observe

Expect stable source IDs, test cases, and chunk-boundary checks. The manifest fixes the corpus and
expected behaviour before a candidate is measured.

### Conclusion

Reproducibility begins with a versioned contract, not with a promising metric after the fact.


## Checkpoint 2 — Objective: run baseline and candidate

**Run:** reuse the real application service with deterministic executors. The candidate is slower
but has the same quality.


In [ ]:
baseline_app = EvaluationApplication(
    HermeticEvaluationExecutor(latency_scale=1.0),
    artifact_dir=artifact_dir,
    metadata_provider=metadata,
)
candidate_app = EvaluationApplication(
    HermeticEvaluationExecutor(latency_scale=1.5),
    artifact_dir=artifact_dir,
    metadata_provider=metadata,
)
baseline = baseline_app.run(manifest, persist=False)
candidate = candidate_app.run(manifest, persist=False)
print("Baseline quality:", baseline["summary"]["quality"])
print("Candidate quality:", candidate["summary"]["quality"])
print("Candidate hard failures:", candidate["errors"]["hard"])


### What to observe

Expect identical perfect quality axes and no hard failures. Latency changes independently from
deterministic quality.

### Conclusion

A baseline and candidate are comparable only because they share schema, corpus, configuration,
and hardware fingerprint.


## Checkpoint 3 — Objective: compare and promote deliberately

**Run:** calculate the conservative verdict, then promote the complete clean candidate atomically.


In [ ]:
comparison = candidate_app.compare(candidate, baseline)
baseline_path = candidate_app.promote(candidate)
print("Verdict:", comparison["verdict"])
print("Quality deltas:", {name: row["delta"] for name, row in comparison["axes"].items()})
print("Promoted:", baseline_path)
assert baseline_path.exists()


### What to observe

Expect `no_clear_change`, zero quality deltas, and a baseline JSON under `/tmp`. Promotion accepts
only a complete, non-partial run from a clean worktree.

### Conclusion

Evaluation closes the loop when evidence—not intuition—controls whether a candidate becomes the
next baseline.

## Optional appendix — real core evaluation

Run `raglab-evaluate run --profile core`, inspect its JSON/Markdown artifacts, compare it with the
approved baseline, then use `raglab-evaluate baseline promote RUN` only after reviewing failures,
compatibility, and latency. PostgreSQL and Ollama remain opt-in service boundaries.
